##Jai Ganesha

# 02 — VisDrone Preprocessing

Goal: Convert raw VisDrone DET annotations into YOLO-format .txt labels,
create train/val/test splits, build a stress_set.json for tiny/crowded scenes,
and save visdrone.yaml for use in YOLO-World fine-tuning.

Inputs  : Raw VisDrone DET folders (images + .txt annotations)
Outputs :
  - processed/visdrone/images/{train,val,test}/
  - processed/visdrone/labels/{train,val,test}/
  - processed/visdrone/visdrone.yaml
  - processed/visdrone/class_dist.csv
  - processed/visdrone/stress_set.json

VisDrone annotation format (each line):
  x_min, y_min, width, height, score, category_id, truncation, occlusion
  → category_id 0 = ignored region (SKIP)
  → category_id 1–11 = actual objects

YOLO format (each line):
  class_id  x_center_norm  y_center_norm  width_norm  height_norm
  (all values normalized 0–1 by image dimensions)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Imports and config load
# ─────────────────────────────────────────────────────────────

import json, os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import cv2

from google.colab import drive
drive.mount('/content/drive')

# ── Load config ───────────────────────────────────────────────
CONFIG_PATH = "/content/drive/MyDrive/DLCV_OV_Analytics/configs/config.json"
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

RAW_VISDRONE      = cfg["raw_visdrone"]
PROC_VISDRONE     = cfg["proc_visdrone"]
METRICS_DIR       = cfg["metrics_dir"]
VIZ_DIR           = cfg["viz_dir"]

PROC_VD_IMG_TRAIN = cfg["proc_vd_img_train"]
PROC_VD_IMG_VAL   = cfg["proc_vd_img_val"]
PROC_VD_IMG_TEST  = cfg["proc_vd_img_test"]
PROC_VD_LBL_TRAIN = cfg["proc_vd_lbl_train"]
PROC_VD_LBL_VAL   = cfg["proc_vd_lbl_val"]
PROC_VD_LBL_TEST  = cfg["proc_vd_lbl_test"]

# ── Class definitions ─────────────────────────────────────────
# VisDrone raw category IDs → YOLO 0-based class IDs
# Raw ID 0 = ignored region → SKIP entirely
# Raw ID 1–11 → YOLO ID 0–10
YOLO_CLASSES = [
    "pedestrian",       # YOLO 0  ← raw 1
    "people",           # YOLO 1  ← raw 2
    "bicycle",          # YOLO 2  ← raw 3
    "car",              # YOLO 3  ← raw 4
    "van",              # YOLO 4  ← raw 5
    "truck",            # YOLO 5  ← raw 6
    "tricycle",         # YOLO 6  ← raw 7
    "awning-tricycle",  # YOLO 7  ← raw 8
    "bus",              # YOLO 8  ← raw 9
    "motor",            # YOLO 9  ← raw 10
    "others",           # YOLO 10 ← raw 11
]

# raw category_id → YOLO class_id mapping
RAW_CAT_TO_YOLO = {raw: raw - 1 for raw in range(1, 12)}

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(VIZ_DIR,     exist_ok=True)

print("✅ Setup complete.")
print(f"   RAW  : {RAW_VISDRONE}")
print(f"   PROC : {PROC_VISDRONE}")
print(f"   YOLO classes ({len(YOLO_CLASSES)}): {YOLO_CLASSES}")

Mounted at /content/drive
✅ Setup complete.
   RAW  : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/VisDrone
   PROC : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone
   YOLO classes (11): ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor', 'others']


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Core VisDrone → YOLO conversion function
# ─────────────────────────────────────────────────────────────

def visdrone_ann_to_yolo(ann_path, img_w, img_h):
    """
    Convert one VisDrone .txt annotation to YOLO format lines.

    VisDrone line: x_min,y_min,width,height,score,category_id,truncation,occlusion
    YOLO line   : class_id x_center_norm y_center_norm width_norm height_norm

    Returns:
        lines (list[str]) : YOLO-format label lines
        stats (dict)      : per-image conversion statistics
    """
    lines = []
    stats = {
        "total_raw"    : 0,
        "skipped_zero" : 0,   # category_id == 0 (ignored region)
        "skipped_degen": 0,   # w <= 0 or h <= 0 before clamping
        "skipped_oob"  : 0,   # w <= 0 or h <= 0 after clamping to image bounds
        "n_small"      : 0,   # boxes with area < 1024 px² (32×32)
        "class_counts" : defaultdict(int),
    }

    with open(ann_path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue

            stats["total_raw"] += 1

            x, y, w, h  = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            category_id = int(parts[5])

            # Skip ignored regions
            if category_id == 0:
                stats["skipped_zero"] += 1
                continue

            # Skip degenerate boxes (zero or negative size)
            if w <= 0 or h <= 0:
                stats["skipped_degen"] += 1
                continue

            # Clamp coordinates to image boundaries
            x = max(0, min(x, img_w - 1))
            y = max(0, min(y, img_h - 1))
            w = min(w, img_w - x)
            h = min(h, img_h - y)

            # Skip if box became degenerate after clamping
            if w <= 0 or h <= 0:
                stats["skipped_oob"] += 1
                continue

            # Convert to YOLO normalized center format
            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w_norm   = w / img_w
            h_norm   = h / img_h

            # Clamp to [0, 1] as a final safety check
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            w_norm   = max(0.0, min(1.0, w_norm))
            h_norm   = max(0.0, min(1.0, h_norm))

            yolo_class = RAW_CAT_TO_YOLO[category_id]
            stats["class_counts"][yolo_class] += 1

            # Flag tiny objects (area < 32×32 pixels)
            if (w * h) < 1024:
                stats["n_small"] += 1

            lines.append(
                f"{yolo_class} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            )

    return lines, stats


# ── Quick test on one sample image ───────────────────────────
available_splits = [
    d for d in sorted(os.listdir(RAW_VISDRONE))
    if os.path.isdir(os.path.join(RAW_VISDRONE, d))
]
train_splits = [d for d in available_splits if "train" in d.lower()]
if not train_splits:
    raise FileNotFoundError(
        f"No train split found in {RAW_VISDRONE}. Found: {available_splits}"
    )

train_split_path = os.path.join(RAW_VISDRONE, train_splits[0])
img_dir_test     = os.path.join(train_split_path, "images")
ann_dir_test     = os.path.join(train_split_path, "annotations")

sample_img_file = sorted(os.listdir(img_dir_test))[0]
sample_ann_file = Path(sample_img_file).stem + ".txt"

img      = cv2.imread(os.path.join(img_dir_test, sample_img_file))
h, w     = img.shape[:2]
lines, stats = visdrone_ann_to_yolo(
    os.path.join(ann_dir_test, sample_ann_file), w, h
)

print(f"Test image : {sample_img_file}  ({w}×{h})")
print(f"Raw boxes  : {stats['total_raw']}")
print(f"YOLO lines : {len(lines)}")
print(f"Skipped  — ignored/degen/oob: "
      f"{stats['skipped_zero']} / {stats['skipped_degen']} / {stats['skipped_oob']}")
print(f"Small objects (< 32×32 px)  : {stats['n_small']}")
print(f"Sample YOLO output:\n  " + "\n  ".join(lines[:3]))

Test image : 0000002_00005_d_0000014.jpg  (960×540)
Raw boxes  : 88
YOLO lines : 83
Skipped  — ignored/degen/oob: 5 / 0 / 0
Small objects (< 32×32 px)  : 59
Sample YOLO output:
  3 0.776042 0.902778 0.077083 0.061111
  3 0.697396 0.829630 0.063542 0.085185
  3 0.652083 0.786111 0.066667 0.094444


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Discover all VisDrone splits dynamically
# Avoids hardcoded folder names that may differ across setups
# ─────────────────────────────────────────────────────────────

available = [
    d for d in sorted(os.listdir(RAW_VISDRONE))
    if os.path.isdir(os.path.join(RAW_VISDRONE, d))
]
print(f"Found {len(available)} split folder(s) in RAW_VISDRONE:")
for d in available:
    print(f"   {d}")

def infer_split_name(folder_name):
    """Map VisDrone folder name to a canonical split name."""
    name = folder_name.lower()
    if "train" in name:
        return "train"
    elif "val" in name:
        return "val"
    elif "test" in name and "challenge" in name:
        return "test_challenge"
    elif "test" in name:
        return "test"
    return None

# Build split map: raw folder name → canonical split name
SPLIT_MAP = {}
for folder in available:
    split = infer_split_name(folder)
    if split:
        SPLIT_MAP[folder] = split
    else:
        print(f"  ⚠️  Unrecognized split folder (skipping): {folder}")

# Map canonical split name → output image/label directories
SPLIT_TO_DIRS = {
    "train"          : (PROC_VD_IMG_TRAIN, PROC_VD_LBL_TRAIN),
    "val"            : (PROC_VD_IMG_VAL,   PROC_VD_LBL_VAL),
    "test"           : (PROC_VD_IMG_TEST,  PROC_VD_LBL_TEST),
    "test_challenge" : (PROC_VD_IMG_TEST,  PROC_VD_LBL_TEST),
}

print(f"\nSplit mapping:")
for raw_name, split in SPLIT_MAP.items():
    img_out, lbl_out = SPLIT_TO_DIRS.get(split, ("?", "?"))
    print(f"   {raw_name}")
    print(f"     → images : {img_out}")
    print(f"     → labels : {lbl_out}")

Found 4 split folder(s) in RAW_VISDRONE:
   VisDrone2019-DET-test-challenge
   VisDrone2019-DET-test-dev
   VisDrone2019-DET-train
   VisDrone2019-DET-val

Split mapping:
   VisDrone2019-DET-test-challenge
     → images : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/images/test
     → labels : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/labels/test
   VisDrone2019-DET-test-dev
     → images : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/images/test
     → labels : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/labels/test
   VisDrone2019-DET-train
     → images : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/images/train
     → labels : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/labels/train
   VisDrone2019-DET-val
     → images : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/images/val
     → labels : /content/drive/MyDr

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: Run conversion for all discovered VisDrone splits
# ─────────────────────────────────────────────────────────────

conversion_stats = {}
stress_set       = []

for raw_split_name, out_split in SPLIT_MAP.items():
    raw_split_path = os.path.join(RAW_VISDRONE, raw_split_name)
    img_in         = os.path.join(raw_split_path, "images")
    ann_in         = os.path.join(raw_split_path, "annotations")

    if not os.path.isdir(img_in) or not os.path.isdir(ann_in):
        print(f"  ⚠️  Skipping {raw_split_name} — missing images/ or annotations/ subfolder")
        continue

    if out_split not in SPLIT_TO_DIRS:
        print(f"  ⚠️  No output dir mapped for split '{out_split}', skipping.")
        continue

    img_out, lbl_out = SPLIT_TO_DIRS[out_split]

    # Output dirs already created by notebook 00 Cell 1, but ensure again
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    img_files = sorted(f for f in os.listdir(img_in) if f.endswith(".jpg"))

    split_stats = {
        "n_images"     : 0,
        "n_boxes"      : 0,
        "n_small"      : 0,
        "skipped_zero" : 0,
        "skipped_degen": 0,
        "skipped_oob"  : 0,
        "class_counts" : defaultdict(int),
    }

    print(f"\nConverting: {raw_split_name} → {out_split}  ({len(img_files)} images)...")

    for img_file in tqdm(img_files, desc=out_split):
        src_img  = os.path.join(img_in,  img_file)
        ann_file = Path(img_file).stem + ".txt"
        src_ann  = os.path.join(ann_in,  ann_file)

        # Skip if annotation file is missing
        if not os.path.exists(src_ann):
            continue

        # Read image to get dimensions (needed for normalization)
        img = cv2.imread(src_img)
        if img is None:
            print(f"  ⚠️  Could not read image: {src_img}")
            continue
        img_h, img_w = img.shape[:2]

        yolo_lines, stats = visdrone_ann_to_yolo(src_ann, img_w, img_h)

        # Copy image to processed folder (raw folder is never modified)
        dst_img = os.path.join(img_out, img_file)
        shutil.copy2(src_img, dst_img)

        # Write YOLO label file (empty file = valid image with no objects)
        dst_lbl = os.path.join(lbl_out, Path(img_file).stem + ".txt")
        with open(dst_lbl, "w") as f:
            f.write("\n".join(yolo_lines))

        # Accumulate statistics
        split_stats["n_images"]      += 1
        split_stats["n_boxes"]       += len(yolo_lines)
        split_stats["n_small"]       += stats["n_small"]
        split_stats["skipped_zero"]  += stats["skipped_zero"]
        split_stats["skipped_degen"] += stats["skipped_degen"]
        split_stats["skipped_oob"]   += stats["skipped_oob"]
        for cls, cnt in stats["class_counts"].items():
            split_stats["class_counts"][cls] += cnt

        # Add to stress set if image has many small objects (hardest cases)
        if stats["n_small"] > 10:
            stress_set.append({
                "image_path": dst_img,
                "label_path": dst_lbl,
                "split"     : out_split,
                "n_boxes"   : len(yolo_lines),
                "n_small"   : stats["n_small"],
            })

    conversion_stats[out_split] = split_stats
    total_skipped = (split_stats["skipped_zero"] +
                     split_stats["skipped_degen"] +
                     split_stats["skipped_oob"])
    print(f"  ✅ {split_stats['n_images']} images")
    print(f"     YOLO labels : {split_stats['n_boxes']:,} boxes")
    print(f"     Skipped     : {total_skipped:,} boxes "
          f"(ignored={split_stats['skipped_zero']:,} / "
          f"degen={split_stats['skipped_degen']:,} / "
          f"oob={split_stats['skipped_oob']:,})")
    print(f"     Small objs  : {split_stats['n_small']:,}")

print(f"\n✅ All splits converted. Stress set: {len(stress_set)} high-difficulty images.")

  ⚠️  Skipping VisDrone2019-DET-test-challenge — missing images/ or annotations/ subfolder

Converting: VisDrone2019-DET-test-dev → test  (1610 images)...


test: 100%|██████████| 1610/1610 [01:28<00:00, 18.13it/s]


  ✅ 1610 images
     YOLO labels : 75,367 boxes
     Skipped     : 2,180 boxes (ignored=2,180 / degen=0 / oob=0)
     Small objs  : 50,915

Converting: VisDrone2019-DET-train → train  (6471 images)...


train: 100%|██████████| 6471/6471 [34:46<00:00,  3.10it/s]


  ✅ 6471 images
     YOLO labels : 344,736 boxes
     Skipped     : 8,814 boxes (ignored=8,813 / degen=1 / oob=0)
     Small objs  : 208,111

Converting: VisDrone2019-DET-val → val  (548 images)...


val: 100%|██████████| 548/548 [02:47<00:00,  3.28it/s]

  ✅ 548 images
     YOLO labels : 38,791 boxes
     Skipped     : 1,378 boxes (ignored=1,378 / degen=0 / oob=0)
     Small objs  : 26,588

✅ All splits converted. Stress set: 5906 high-difficulty images.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Save visdrone.yaml, stress_set.json, class_dist.csv
# ─────────────────────────────────────────────────────────────

import yaml

# ── 1. visdrone.yaml — used by YOLO-World/Ultralytics for fine-tuning ────
yaml_content = {
    "path"  : PROC_VISDRONE,
    "train" : "images/train",
    "val"   : "images/val",
    "test"  : "images/test",
    "nc"    : len(YOLO_CLASSES),
    "names" : YOLO_CLASSES,
}
yaml_path = os.path.join(PROC_VISDRONE, "visdrone.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
print(f"✅ visdrone.yaml saved: {yaml_path}")

# ── 2. stress_set.json — high-difficulty images for targeted evaluation ───
stress_path = os.path.join(PROC_VISDRONE, "stress_set.json")
with open(stress_path, "w") as f:
    json.dump(stress_set, f, indent=2)
print(f"✅ stress_set.json: {len(stress_set)} images → {stress_path}")

# ── 3. class_dist.csv — class distribution per split ─────────────────────
rows = []
for split, stats in conversion_stats.items():
    for cls_id, count in stats["class_counts"].items():
        rows.append({
            "split"     : split,
            "class_id"  : cls_id,
            "class_name": YOLO_CLASSES[cls_id],
            "count"     : count,
        })

df_dist   = pd.DataFrame(rows)
dist_path = os.path.join(PROC_VISDRONE, "class_dist.csv")
df_dist.to_csv(dist_path, index=False)
print(f"✅ class_dist.csv saved: {dist_path}")

# ── Print pivot table ─────────────────────────────────────────
pivot = df_dist.pivot_table(
    index="class_name", columns="split",
    values="count", aggfunc="sum", fill_value=0
)
print("\nClass Distribution across splits:")
print(pivot.to_string())

✅ visdrone.yaml saved: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/visdrone.yaml
✅ stress_set.json: 5906 images → /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/stress_set.json
✅ class_dist.csv saved: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/class_dist.csv

Class Distribution across splits:
split             test   train    val
class_name                           
awning-tricycle    599    3246    532
bicycle           1302   10480   1287
bus               2940    5926    251
car              28074  144866  14064
motor             5845   29647   4886
others             265    1532     32
pedestrian       21006   79337   8844
people            6376   27059   5125
tricycle           530    4812   1045
truck             2659   12875    750
van               5771   24956   1975


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: Visualize sample YOLO-format annotations (verification)
# Reads from processed folder to confirm conversion is correct
# ─────────────────────────────────────────────────────────────

COLOR_MAP = {
    0: (255, 80,  80),   # pedestrian  — red
    1: (255, 160, 80),   # people      — orange
    2: (80,  180, 80),   # bicycle     — green
    3: (80,  120, 255),  # car         — blue
    4: (180, 80,  255),  # van         — purple
    5: (255, 220, 60),   # truck       — yellow
    6: (80,  220, 220),  # tricycle    — cyan
    7: (200, 120, 80),   # awning-tri  — brown
    8: (255, 100, 180),  # bus         — pink
    9: (100, 200, 100),  # motor       — light green
    10:(180, 180, 180),  # others      — gray
}

def draw_yolo_labels(img, label_path):
    """Draw YOLO-format bounding boxes on an image (in-place)."""
    h, w = img.shape[:2]
    if not os.path.exists(label_path):
        return img, 0
    box_count = 0
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id                    = int(parts[0])
            x_center, y_center, bw, bh = map(float, parts[1:])
            # Convert back to pixel coordinates for drawing
            x1 = int((x_center - bw / 2) * w)
            y1 = int((y_center - bh / 2) * h)
            x2 = int((x_center + bw / 2) * w)
            y2 = int((y_center + bh / 2) * h)
            color = COLOR_MAP.get(cls_id, (200, 200, 200))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)
            label = YOLO_CLASSES[cls_id] if cls_id < len(YOLO_CLASSES) else str(cls_id)
            cv2.putText(img, label, (x1, max(0, y1 - 3)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)
            box_count += 1
    return img, box_count

# ── Plot 6 samples from processed train folder ───────────────
proc_img_dir = PROC_VD_IMG_TRAIN
proc_lbl_dir = PROC_VD_LBL_TRAIN

sample_files = sorted(os.listdir(proc_img_dir))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, img_file in enumerate(sample_files):
    img_path = os.path.join(proc_img_dir, img_file)
    lbl_path = os.path.join(proc_lbl_dir, Path(img_file).stem + ".txt")

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img, n_boxes = draw_yolo_labels(img, lbl_path)

    axes[i].imshow(img)
    axes[i].set_title(f"{img_file}\n{n_boxes} boxes", fontsize=8)
    axes[i].axis("off")

plt.suptitle("VisDrone — Processed YOLO Labels (Verification)", fontsize=13)
plt.tight_layout()
save_path = os.path.join(VIZ_DIR, "visdrone_yolo_labels_check.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {save_path}")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7: Final preprocessing summary
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("  VISDRONE PREPROCESSING SUMMARY")
print("=" * 60)

for split, stats in conversion_stats.items():
    img_out, lbl_out = SPLIT_TO_DIRS.get(split, ("?", "?"))
    n_imgs_on_disk   = len(os.listdir(img_out)) if os.path.isdir(img_out) else 0
    n_lbls_on_disk   = len(os.listdir(lbl_out)) if os.path.isdir(lbl_out) else 0

    print(f"\n  Split: {split}")
    print(f"    Images written    : {n_imgs_on_disk}")
    print(f"    Labels written    : {n_lbls_on_disk}")
    print(f"    Total YOLO boxes  : {stats['n_boxes']:,}")
    print(f"    Small objects     : {stats['n_small']:,}  "
          f"({100*stats['n_small']/max(stats['n_boxes'],1):.1f}% of boxes)")
    top3 = sorted(stats["class_counts"].items(), key=lambda x: -x[1])[:3]
    print(f"    Top 3 classes     : "
          f"{[(YOLO_CLASSES[k], v) for k, v in top3]}")

print(f"\n  Stress set          : {len(stress_set)} high-difficulty images")
print(f"\n  Output files:")
print(f"    visdrone.yaml    : {os.path.join(PROC_VISDRONE, 'visdrone.yaml')}")
print(f"    stress_set.json  : {os.path.join(PROC_VISDRONE, 'stress_set.json')}")
print(f"    class_dist.csv   : {os.path.join(PROC_VISDRONE, 'class_dist.csv')}")
print(f"    viz sample       : {os.path.join(VIZ_DIR, 'visdrone_yolo_labels_check.png')}")
print("\n✅ Notebook 02 complete. Ready for Notebook 03.")

  VISDRONE PREPROCESSING SUMMARY

  Split: test
    Images written    : 1610
    Labels written    : 1610
    Total YOLO boxes  : 75,367
    Small objects     : 50,915  (67.6% of boxes)
    Top 3 classes     : [('car', 28074), ('pedestrian', 21006), ('people', 6376)]

  Split: train
    Images written    : 6471
    Labels written    : 6471
    Total YOLO boxes  : 344,736
    Small objects     : 208,111  (60.4% of boxes)
    Top 3 classes     : [('car', 144866), ('pedestrian', 79337), ('motor', 29647)]

  Split: val
    Images written    : 548
    Labels written    : 548
    Total YOLO boxes  : 38,791
    Small objects     : 26,588  (68.5% of boxes)
    Top 3 classes     : [('car', 14064), ('pedestrian', 8844), ('people', 5125)]

  Stress set          : 5906 high-difficulty images

  Output files:
    visdrone.yaml    : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone/visdrone.yaml
    stress_set.json  : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/vis